# HikeLogic — DPO on top of the SFT model

Trains a LoRA adapter via DPO on human-ranked answer pairs, merges, pushes to HF Hub.

**Workflow:**
1. Generate pairs offline: `python -m finetune.build_dpo_candidates ...`
2. Reviewer fills `chosen` ∈ {`a`, `b`} per row in the resulting JSONL.
3. This notebook converts → DPO format → trains → merges → pushes.

**Runtime:** Colab Pro + A100, ~1 h on ~50 pairs.

## 1. Install

In [ ]:
!pip install -q "transformers==4.51.3" "trl>=0.13,<0.21" "peft>=0.13" "bitsandbytes>=0.43" "datasets>=2.20" "accelerate>=0.34" huggingface_hub

## 2. HF login + upload ranked JSONL

`finetune/data/dpo_ranked.jsonl`: one row per pair with `prompt`, `answer_a`, `answer_b`, `chosen` ∈ {`a`,`b`}.

In [ ]:
from huggingface_hub import login
login()

## 3. Convert ranked pairs → DPO dataset

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

RANKED_PATH = Path("/content/HikeLogic/finetune/data/dpo_ranked.jsonl")

rows = []
for line in RANKED_PATH.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    r = json.loads(line)
    if r.get("chosen") not in {"a", "b"}:
        continue  # skip unranked
    chosen  = r["answer_a"] if r["chosen"] == "a" else r["answer_b"]
    rejected = r["answer_b"] if r["chosen"] == "a" else r["answer_a"]
    rows.append({"prompt": r["prompt"], "chosen": chosen, "rejected": rejected})

print(f"Ranked pairs: {len(rows)}")
assert rows, "No ranked pairs found — fill in the 'chosen' column before training."
ds = Dataset.from_list(rows).train_test_split(test_size=0.1, seed=42)
ds

## 4. Train

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer

BASE = "edededi/hikelogic-qwen2.5-7b"
OUT  = "/content/outputs/dpo"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = "<|endoftext|>" if "<|endoftext|>" in tokenizer.get_vocab() else tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

args = DPOConfig(
    output_dir=OUT,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    beta=0.1,
    max_length=2048,
    max_prompt_length=1600,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.train()
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)

## 5. Merge + push DPO model

In [ ]:
from peft import PeftModel

DPO_REPO = "edededi/hikelogic-qwen2.5-7b-dpo"

base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16)
merged = PeftModel.from_pretrained(base, OUT).merge_and_unload()
merged.push_to_hub(DPO_REPO)
tokenizer.push_to_hub(DPO_REPO)
print(f"Done: https://huggingface.co/{DPO_REPO}")